# PrepAI Custom Models Training

This notebook trains the 3 core custom ML models for PrepAI without external APIs.

**IMPORTANT:** Make sure you are using a GPU runtime.
Go to `Runtime > Change runtime type` and select `T4 GPU`.

In [ ]:
# Install dependencies and remove torchvision (fixes Colab VideoReader bug)
!pip uninstall -y torchvision
!pip install transformers datasets accelerate torch scikit-learn pandas sentence-transformers

## Model 1: Answer Quality Classifier (BERT)
Trains a BERT model to classify answer quality into 4 classes: Poor, Average, Good, Excellent.

In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("GPU Available for BERT:", torch.cuda.is_available())

# Dummy dataset - replace with your actual CSV reading logic
data_bert = {
    "question": ["What is OOP?", "Explain polymorphism.", "What is a database?", "How does DNS work?", "Explain REST API."] * 200,
    "answer": [
        "It is object oriented programming.", # 0
        "Polymorphism means many forms, like method overloading.", # 1
        "A database is an organized collection of data, typically stored electronically.", # 2
        "DNS resolves domain names to IP addresses by querying nameservers sequentially.", # 3
        "REST is an API." # 0
    ] * 200,
    "label": [0, 1, 2, 3, 0] * 200
}
df_bert = pd.DataFrame(data_bert)
df_bert['text'] = "Question: " + df_bert['question'] + " Answer: " + df_bert['answer']

train_df_b, val_df_b = train_test_split(df_bert, test_size=0.2, random_state=42)
train_dataset_b = Dataset.from_pandas(train_df_b)
val_dataset_b = Dataset.from_pandas(val_df_b)

tokenizer_bert = BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize_bert(examples):
    return tokenizer_bert(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset_b = train_dataset_b.map(tokenize_bert, batched=True)
val_dataset_b = val_dataset_b.map(tokenize_bert, batched=True)
train_dataset_b.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_b.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

model_bert = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

def compute_metrics_bert(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args_bert = TrainingArguments(
    output_dir="./results_bert", num_train_epochs=3, per_device_train_batch_size=16,
    per_device_eval_batch_size=16, warmup_steps=100, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", learning_rate=2e-5, load_best_model_at_end=True
)

trainer_bert = Trainer(model=model_bert, args=training_args_bert, train_dataset=train_dataset_b, eval_dataset=val_dataset_b, compute_metrics=compute_metrics_bert)
print("Starting BERT Training...")
trainer_bert.train()

print("Saving BERT Model...")
model_bert.save_pretrained("./answer_quality_model")
tokenizer_bert.save_pretrained("./answer_quality_model")
print("Model 1 Saved Successfully!")

## Model 2: Interview Question Generator (GPT-2)
Trains GPT-2 to generate the next interview question based on role, topic, and previous answers.

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

print("GPU Available for GPT-2:", torch.cuda.is_available())

texts_gpt = [
    "[ROLE] Software Engineer [TOPIC] OOP [ANSWER] None [Q_SCORE] None [NEXT_Q] Explain the four pillars of OOP.",
    "[ROLE] Software Engineer [TOPIC] OOP [ANSWER] It is inheritance and polymorphism [Q_SCORE] 1 [NEXT_Q] Can you elaborate on Encapsulation and Abstraction?",
    "[ROLE] Data Analyst [TOPIC] SQL [ANSWER] I use select star to get data [Q_SCORE] 1 [NEXT_Q] How would you optimize a slow running query?",
    "[ROLE] Data Analyst [TOPIC] SQL [ANSWER] I use indexes and explain plan [Q_SCORE] 3 [NEXT_Q] Excellent. Tell me the difference between a clustered and non-clustered index."
] * 250

dataset_gpt = Dataset.from_dict({"text": texts_gpt})

tokenizer_gpt = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
model_gpt = GPT2LMHeadModel.from_pretrained("gpt2")

def tokenize_gpt(examples):
    tokenized = tokenizer_gpt(examples["text"], truncation=True, max_length=128, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_datasets_gpt = dataset_gpt.map(tokenize_gpt, batched=True, remove_columns=["text"])
split_gpt = tokenized_datasets_gpt.train_test_split(test_size=0.1, seed=42)
train_dataset_gpt = split_gpt["train"]
val_dataset_gpt = split_gpt["test"]

data_collator_gpt = DataCollatorForLanguageModeling(tokenizer=tokenizer_gpt, mlm=False)

training_args_gpt = TrainingArguments(
    output_dir="./results_gpt2", num_train_epochs=4,
    per_device_train_batch_size=8, per_device_eval_batch_size=8, eval_steps=100,
    save_steps=100, warmup_steps=50, learning_rate=5e-5, eval_strategy="steps"
)

trainer_gpt = Trainer(model=model_gpt, args=training_args_gpt, train_dataset=train_dataset_gpt, eval_dataset=val_dataset_gpt, data_collator=data_collator_gpt)
print("Starting GPT-2 Training...")
trainer_gpt.train()

print("Saving GPT-2 Model...")
model_gpt.save_pretrained("./question_generator_model")
tokenizer_gpt.save_pretrained("./question_generator_model")
print("Model 2 Saved Successfully!")

## Model 3: Semantic Answer Relevance (Sentence-BERT)
Trains SBERT to compute relevance (0.0 to 1.0) between the interview question and student's answer.

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

print("GPU Available for SBERT:", torch.cuda.is_available())

data_sbert = {
    "question": ["What is OOP?", "What is OOP?", "Explain polymorphism.", "Explain polymorphism.", "What is a database?", "What is a database?"] * 150,
    "answer": [
        "Object-oriented programming is a paradigm based on objects.", # High
        "It is a way to write code.", # Medium
        "Polymorphism allows objects of different types to be treated as instances of the same class.", # High
        "It means poly and morph.", # Low
        "A structured set of data held in a computer.", # High
        "A place to store things like clothes." # Low
    ] * 150,
    "score": [1.0, 0.5, 1.0, 0.1, 0.9, 0.0] * 150
}
df_sbert = pd.DataFrame(data_sbert)

train_examples = []
for _, row in df_sbert.iterrows():
    train_examples.append(InputExample(texts=[row['question'], row['answer']], label=float(row['score'])))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

model_sbert = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.CosineSimilarityLoss(model_sbert)

print("Starting SBERT Training...")
model_sbert.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4, warmup_steps=100, show_progress_bar=True
)

print("Saving SBERT Model...")
model_sbert.save("./semantic_relevance_model")
print("Model 3 Saved Successfully!")

## Download the Models
Run the cell below to zip all the models. Then you can download `prepai_trained_models.zip` from the Colab file explorer on the left and extract it into your project folder.

In [ ]:
!mkdir -p prepai_trained_models
!mv answer_quality_model prepai_trained_models/
!mv question_generator_model prepai_trained_models/
!mv semantic_relevance_model prepai_trained_models/

!zip -r prepai_trained_models.zip prepai_trained_models/

from google.colab import files
files.download('prepai_trained_models.zip')